In [ ]:
import numpy as np
from collections import Counter
from sklearn.tree import DecisionTreeClassifier

class RandomForestFromScratch:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            X_sample, y_sample = self._bootstrap_samples(X, y)

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.n_features # This picks random features at each split
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        # Randomly pick indices with replacement
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X.iloc[indices], y.iloc[indices]

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        # Aggregation: Majority Vote
        # tree_preds shape is (n_trees, n_test_samples), swap it to iterate over samples
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        predictions = [Counter(sample_preds).most_common(1)[0][0] for sample_preds in tree_preds]
        return np.array(predictions)

In [ ]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("/content/Heart Attack.csv")
dataset = dataset[['impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin', 'class']]
dataset['class'] = dataset['class'].map({'positive': 1, 'negative': 0})


train = dataset.sample(frac=0.8, random_state=0)
test = dataset.drop(train.index) # remaining 20%

X_ori = train.iloc[:, :-1]
X_ori_mean = X_ori.mean(axis=0)
X_ori_std  = X_ori.std(axis=0)
X_norm = (X_ori - X_ori_mean) / X_ori_std
y_target = train.iloc[:, -1]

X_train = X_norm
y_train = y_target


X_test_ori = test.iloc[:, :-1]
y_test = test.iloc[:, -1]
# Normalize test features using TRAIN mean & std
X_test = (X_test_ori - X_ori_mean) / X_ori_std


# Train our forest
rf = RandomForestFromScratch(n_trees=5, n_features="sqrt")
rf.fit(X_train, y_train)

# Check accuracy
predictions = rf.predict(X_test)
accuracy = np.sum(predictions == y_test) / len(y_test)
print(f"Random Forest Accuracy: {accuracy * 100:.2f}%")

Random Forest Accuracy: 97.73%
